# Phase 8 — Classical ML: Rules + Isolation Forest

This notebook implements and evaluates two detectors on a held-out temporal
test set:
1. **Rules engine** — deterministic thresholds (no training)
2. **Isolation Forest** — scikit-learn unsupervised model trained ONLY on
   normal traces from the temporal train split

**Pre-registration (committed before any modeling):**
- Target metric: PR-AUC (primary), F1 at fixed threshold (secondary)
- FP ceiling: <= 10% false-positive rate on held-out test set
- See `docs/gate_c_preregistration.md` for full pre-registration text.

**Feature set:** 13 pruned features from `data/features/v2/features.parquet`.
Dropped: 4 agent-level baselines (MVP 1 placeholder) and `calls_so_far`
(perfectly correlated with `call_position`).


In [ ]:
from __future__ import annotations

import json
import warnings
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import (
    auc,
    average_precision_score,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["figure.dpi"] = 100
plt.show = lambda *args, **kwargs: None

from agentsentinel.data.split import load_features, temporal_split
from agentsentinel.detectors.isoforest import (
    get_isoforest_metadata,
    predict_isolation_forest,
    train_isolation_forest,
)
from agentsentinel.detectors.rules import apply_rules, get_rules_metadata

FEATURES = [
    "agent_task_latency_zscore",
    "agent_task_retry_zscore",
    "agent_task_token_zscore",
    "agent_task_cost_zscore",
    "task_latency_zscore",
    "task_retry_zscore",
    "task_token_zscore",
    "task_cost_zscore",
    "call_position",
    "time_since_last_call_ms",
    "tool_repeat_in_session",
    "permission_match",
    "trace_event_count",
]

print("Loading data and performing temporal split...")
df = load_features()
train, test, cutoff_time = temporal_split(df, train_frac=0.8)
print(f"Train: {train['trace_id'].nunique()} traces, {len(train)} events")
print(f"Test: {test['trace_id'].nunique()} traces, {len(test)} events")
print(f"Cutoff time: {cutoff_time}")


## 1. Rules Engine

**Threshold justification:**
- `tool_repeat_in_session >= 3`: 3+ repetitions of the same tool within a session strongly suggests a loop rather than legitimate repeated calls.
- `task_retry_zscore >= 2.0`: 2 standard deviations above task-type mean retry count signals a retry storm.
- `task_token_zscore >= 2.5` or `task_cost_zscore >= 2.5`: Resource spike relative to task-type baseline. Slightly higher threshold because resource variance is naturally larger.


In [ ]:
rules_meta = get_rules_metadata()
print("Rules thresholds:")
for k, v in rules_meta.items():
    if k.endswith("_threshold"):
        print(f"  {k}: {v}")
    elif k.endswith("_justification"):
        print(f"    -> {v}")

test_with_rules = apply_rules(test.copy())
print(f"\nRules flagged {test_with_rules['rules_anomalous'].sum()} events as anomalous out of {len(test_with_rules)}")


## 2. Isolation Forest

**Training protocol:** Isolation Forest is trained ONLY on normal-labeled traces
from the temporal train split. Anomalous traces are validation/test only.


In [ ]:
iso_meta = get_isoforest_metadata()
print("Isolation Forest configuration:")
for k, v in iso_meta.items():
    if k != "features":
        print(f"  {k}: {v}")

model = train_isolation_forest(train)
test_with_iso = predict_isolation_forest(model, test.copy())
print(f"\nIsolation Forest flagged {test_with_iso['iso_anomaly'].sum()} events as anomalous out of {len(test_with_iso)}")


## 3. Evaluation: Overall Metrics

Compares both detectors on precision, recall, F1, AUROC, PR-AUC, and
false-positive rate.


In [ ]:
def compute_metrics(y_true: pd.Series, y_pred: pd.Series, scores: pd.Series) -> dict:
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

    auroc = roc_auc_score(y_true, scores) if len(y_true.unique()) > 1 else 0.0
    pr_auc = average_precision_score(y_true, scores) if len(y_true.unique()) > 1 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auroc": auroc,
        "pr_auc": pr_auc,
        "fpr": fpr,
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

y_true = (test["label"] == "anomalous").astype(int)

rules_metrics = compute_metrics(
    y_true,
    test_with_rules["rules_anomalous"].astype(int),
    test_with_rules["rules_anomalous"].astype(float),
)

iso_metrics = compute_metrics(
    y_true,
    test_with_iso["iso_anomaly"].astype(int),
    test_with_iso["iso_raw_score"],
)

print("\n" + "="*60)
print("GATE C EVALUATION RESULTS")
print("="*60)
print("\nRules Engine:")
for k, v in rules_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\nIsolation Forest:")
for k, v in iso_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

print("\n" + "="*60)
print("GATE C PASS/FAIL")
print("="*60)
gate_c_pass = iso_metrics["pr_auc"] > 0.5 and iso_metrics["fpr"] <= 0.10
print(f"\nPR-AUC target: > 0.5 (baseline for imbalanced binary classification)")
print(f"Isolation Forest PR-AUC: {iso_metrics['pr_auc']:.4f}")
print(f"FP ceiling: <= 10%")
print(f"Isolation Forest FPR: {iso_metrics['fpr']:.4f}")
print(f"\nGate C: {'PASS' if gate_c_pass else 'FAIL'}")
print(f"\nActual (not hypothetical) — Isolation Forest vs Rules:")
print(f"  PR-AUC: {iso_metrics['pr_auc']:.4f} vs {rules_metrics['pr_auc']:.4f}")
print(f"  FPR: {iso_metrics['fpr']:.4f} vs {rules_metrics['fpr']:.4f}")
print(f"  F1: {iso_metrics['f1']:.4f} vs {rules_metrics['f1']:.4f}")


## 4. Per-Anomaly-Type Evaluation

Breaks down performance by `tool_loop`, `retry_storm`, and `resource_spike`
to see if the naive pooled approach loses category-specific signal.


In [ ]:
def evaluate_by_anomaly_type(df: pd.DataFrame, detector_col: str, score_col: str) -> pd.DataFrame:
    results = []
    for anomaly_type in ["tool_loop", "retry_storm", "resource_spike"]:
        subset = df[df["anomaly_type"] == anomaly_type].copy()
        if len(subset) == 0:
            continue

        y_true = (subset["label"] == "anomalous").astype(int)
        y_pred = subset[detector_col].astype(int)
        scores = subset[score_col]

        tp = ((y_true == 1) & (y_pred == 1)).sum()
        fp = ((y_true == 0) & (y_pred == 1)).sum()
        tn = ((y_true == 0) & (y_pred == 0)).sum()
        fn = ((y_true == 1) & (y_pred == 0)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
        pr_auc = average_precision_score(y_true, scores) if len(y_true.unique()) > 1 else 0.0

        results.append({
            "anomaly_type": anomaly_type,
            "count": len(subset),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "fpr": fpr,
            "pr_auc": pr_auc,
        })

    return pd.DataFrame(results)

print("\nPer-anomaly-type performance (Isolation Forest):")
iso_by_type = evaluate_by_anomaly_type(test_with_iso, "iso_anomaly", "iso_raw_score")
print(iso_by_type.to_string(index=False))

print("\nPer-anomaly-type performance (Rules):")
rules_by_type = evaluate_by_anomaly_type(test_with_rules, "rules_anomalous", "rules_anomalous")
print(rules_by_type.to_string(index=False))

print("\nNaive pooled-feature hypothesis test:")
print("If Isolation Forest underperforms on tool_loop/retry_storm relative to")
print("resource_spike, this reproduces the Trajectory Guard finding that pooling")
print("dilutes sequential anomaly signal.")


## 5. Detection Latency

Reports p50 and p95 detection latency (time from trace start to first flag).


In [ ]:
def compute_detection_latency(df: pd.DataFrame, flag_col: str) -> dict:
    flagged = df[df[flag_col] == True].copy()
    if len(flagged) == 0:
        return {"p50_ms": None, "p95_ms": None, "mean_ms": None}

    flagged["trace_start"] = flagged.groupby("trace_id")["timestamp"].transform("min")
    flagged["latency_ms"] = (flagged["timestamp"] - flagged["trace_start"]).dt.total_seconds() * 1000

    latencies = flagged["latency_ms"]
    return {
        "p50_ms": latencies.median(),
        "p95_ms": latencies.quantile(0.95),
        "mean_ms": latencies.mean(),
    }

print("\nDetection latency:")
print("\nRules:")
rules_lat = compute_detection_latency(test_with_rules, "rules_anomalous")
for k, v in rules_lat.items():
    print(f"  {k}: {v:.1f}" if v is not None else f"  {k}: N/A")

print("\nIsolation Forest:")
iso_lat = compute_detection_latency(test_with_iso, "iso_anomaly")
for k, v in iso_lat.items():
    print(f"  {k}: {v:.1f}" if v is not None else f"  {k}: N/A")


## 6. Phase 8 Findings

- **Rules engine:** Precision 1.000, recall 0.394, F1 0.566, PR-AUC 0.574, FPR 0.000.
  Zero false positives but low recall — misses many sequential anomalies.
- **Isolation Forest:** Precision 0.364, recall 0.837, F1 0.507, PR-AUC 0.195, FPR 0.618.
  High recall but PR-AUC below baseline and FPR far exceeds the 10% ceiling.
- **Gate C result:** FAIL. Both pre-registered targets missed:
  - PR-AUC 0.195 < 0.5 target
  - FPR 61.8% > 10% ceiling
- **Per-anomaly-type performance (Isolation Forest):**
  - tool_loop (48 test): F1 0.968, recall 0.938
  - retry_storm (30 test): F1 0.929, recall 0.867
  - resource_spike (26 test): F1 0.762, recall 0.615
  All anomaly types have >= 26 test examples (above the ~20-30 floor).
- **Naive pooled-feature hypothesis:** Isolation Forest does NOT underperform on
  sequential anomaly types relative to resource_spike. It performs BETTER on
  tool_loop and retry_storm than on resource_spike. This does NOT reproduce
  the Trajectory Guard finding that pooling dilutes sequential anomaly signal.
- **Next steps:** Per Master Prompt, Phase 9 proceeds with rules as the
  operational baseline. Isolation Forest is documented as non-deployed with
  reasons: PR-AUC below baseline and FPR ceiling violation on held-out test set.

**Reproducibility metadata:**
- Random seed: 42
- scikit-learn version: recorded at training time
- Dataset version: data/features/v3/features.parquet
- Hardware: CPU-only
- Split: temporal 80/20, cutoff 2026-01-02T09:48:00+00:00
- Train: 496 traces (97 anomalous, 399 normal), 1473 events
- Test: 124 traces (23 anomalous, 101 normal), 350 events

**Note:** These results supersede the earlier Phase 8 report (PR-AUC 0.967/0.991,
FPR 66.7%/0%). The earlier numbers were invalidated by a Phase 2 data-generation
bug where normal and anomalous traces were generated in separate batches with
sequential timestamps, causing the temporal split to trivially separate classes.
See docs/design_decisions.md for details.
